# Intelligent Agents and Rationality

## 📚 Learning Objectives

By completing this notebook, you will:
- Describe agents, environments, and rationality
- Classify agents by percepts, actions, and goals

## 🔗 Prerequisites

- ✅ Basic Python
- ✅ Basic NumPy/Pandas (when applicable)

---

---

# Intelligent Agents and Rationality
## AIAT 111 - Introduction to AI

---

## 📚 Learning Objectives

This notebook demonstrates key concepts through hands-on examples.

By completing this notebook, you will:
- Understand intelligent agents and their components
- Define rationality in AI systems
- Implement simple intelligent agents
- Analyze agent decision-making processes

---

## 🔗 Prerequisites

- ✅ Python 3.8+ installed
- ✅ Required libraries (see `requirements.txt`)
- ✅ Basic Python knowledge

---

## Real-World Context

You are designing an intelligent agent for a smart home system that needs to make rational decisions based on sensor inputs and user preferences.

---

## Defining Intelligent Agents

Understand what makes an agent intelligent and the key components.


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# A complete intelligent agent in ~50 lines: sensors, a knowledge base, and actuators
# wired into the classic perceive -> think -> act loop.
# This loop is THE mental model for every agent in this course, from vacuum bots to game AIs.
# Setup
print('✅ Setup complete!')


class IntelligentAgent:
    """A simple intelligent agent that runs a perceive -> think -> act loop.

    Here the agent is a smart-home climate controller:
    - sensors read the room state (only what the sensors can measure!),
    - the knowledge base holds the user's comfort preferences,
    - actuators switch devices on or off.
    """

    def __init__(self, sensors, actuators, knowledge_base):
        self.sensors = sensors
        self.actuators = actuators
        self.knowledge_base = knowledge_base

    def perceive(self, environment):
        """Agent perceives the environment through its sensors.

        Returns only the values the agent's sensors can measure — the rest
        of the environment stays invisible to the agent.
        """
        return {key: environment[key] for key in self.sensors if key in environment}

    def think(self, perception):
        """Agent compares percepts against its knowledge base and picks an action."""
        prefs = self.knowledge_base
        temperature = perception.get('temperature')
        occupied = perception.get('occupancy')

        if not occupied:
            return 'eco_mode'  # nobody home: save energy
        if temperature > prefs['max_comfort_temp']:
            return 'cooling_on'
        if temperature < prefs['min_comfort_temp']:
            return 'heating_on'
        return 'hold'

    def act(self, action):
        """Agent acts on the environment through an actuator."""
        device = {
            'cooling_on': 'AC unit',
            'heating_on': 'heater',
            'eco_mode': 'thermostat',
            'hold': 'thermostat',
        }[action]
        return f"{device} <- {action}"


# Build one concrete agent: a smart-home climate controller with two sensors and simple comfort rules.
agent = IntelligentAgent(
    sensors=['temperature', 'occupancy'],
    actuators=['AC unit', 'heater', 'thermostat'],
    knowledge_base={'min_comfort_temp': 20, 'max_comfort_temp': 26},
)

print('Agent created. Running the perceive -> think -> act loop:\n')

# Note: 'humidity' is in the environment but NOT among the agent's sensors,
# so it never appears in the perception — agents only know what they sense.
environments = [
    {'temperature': 31, 'occupancy': True, 'humidity': 60},
    {'temperature': 24, 'occupancy': True, 'humidity': 45},
    {'temperature': 17, 'occupancy': True, 'humidity': 50},
    {'temperature': 33, 'occupancy': False, 'humidity': 55},
]

# Run the perceive -> think -> act loop over four different room states and watch the decisions change.
for env in environments:
    perception = agent.perceive(env)
    action = agent.think(perception)
    result = agent.act(action)
    print(f"Environment: {env}")
    print(f"  Perceived (sensors only): {perception}")
    print(f"  Decision:  {action}")
    print(f"  Actuation: {result}\n")

✅ Setup complete!
Agent created. Running the perceive -> think -> act loop:

Environment: {'temperature': 31, 'occupancy': True, 'humidity': 60}
  Perceived (sensors only): {'temperature': 31, 'occupancy': True}
  Decision:  cooling_on
  Actuation: AC unit <- cooling_on

Environment: {'temperature': 24, 'occupancy': True, 'humidity': 45}
  Perceived (sensors only): {'temperature': 24, 'occupancy': True}
  Decision:  hold
  Actuation: thermostat <- hold

Environment: {'temperature': 17, 'occupancy': True, 'humidity': 50}
  Perceived (sensors only): {'temperature': 17, 'occupancy': True}
  Decision:  heating_on
  Actuation: heater <- heating_on

Environment: {'temperature': 33, 'occupancy': False, 'humidity': 55}
  Perceived (sensors only): {'temperature': 33, 'occupancy': False}
  Decision:  eco_mode
  Actuation: thermostat <- eco_mode



## Rationality in AI

Understand rationality and how agents make rational decisions.


In [2]:
# Rationality: a rational agent chooses the action with the highest EXPECTED utility.
# Expected utility of an action = sum over outcomes of P(outcome) x utility(outcome).

def rational_decision(actions, utilities):
    """Return the action with the highest expected utility.

    Args:
        actions: list of action names
        utilities: dict mapping action -> list of (probability, utility) outcomes

    Returns:
        (best_action, expected_utilities_dict)
    """
    expected = {}
    for action in actions:
        expected[action] = sum(p * u for p, u in utilities[action])
    best_action = max(expected, key=expected.get)
    return best_action, expected


# Smart-home example: should the agent pre-cool the house before the
# afternoon heat, given an uncertain weather forecast?
# Forecast: 70% chance of a hot afternoon, 30% chance it stays mild.
actions = ['pre_cool_now', 'wait_and_see']
utilities = {
    'pre_cool_now': [(0.7, +8), (0.3, -2)],   # comfortable if hot; wasted energy if mild
    'wait_and_see': [(0.7, -5), (0.3, +3)],   # uncomfortable scramble if hot; fine if mild
}

best, expected = rational_decision(actions, utilities)

print('Expected utility of each action:')
for action, eu in expected.items():
    print(f'  {action:14s} -> {eu:+.2f}')

print(f'\nRational choice: {best} (highest expected utility)')
print('\nNote: rationality means the best decision GIVEN the probabilities the')
print('agent knows — not a guarantee of the best outcome in every single case.')


Expected utility of each action:
  pre_cool_now   -> +5.00
  wait_and_see   -> -2.60

Rational choice: pre_cool_now (highest expected utility)

Note: rationality means the best decision GIVEN the probabilities the
agent knows — not a guarantee of the best outcome in every single case.


---

## ✅ Summary

**What you implemented and ran:**

- An **intelligent agent** with a working `perceive -> think -> act` loop: the
  smart-home agent read only what its sensors expose (humidity was in the
  environment but never perceived), matched percepts against its knowledge base,
  and commanded its actuators (cooling, heating, eco mode, hold).
- A **rational decision** function that computes expected utility for each action
  and picks the maximum — in the pre-cooling example the printed expected
  utilities decide the choice, and rationality is about the best *expected*
  decision, not a guaranteed best outcome.

**Next:** `05_philosophy_turing_test.ipynb` asks what it would even mean for
such an agent to "think"; `12_case_studies_intelligent_agents.ipynb` applies
the agent loop to more case studies.


## 📚 References

1. Wooldridge, M., & Jennings, N. R. (1995). *Intelligent Agents: Theory and Practice*. The Knowledge Engineering Review, 10(2), 115–152.
2. Russell, S., & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4th ed.), Ch. 2: Intelligent Agents. Pearson.
3. Wang, L., Ma, C., Feng, X., et al. (2023). *A Survey on Large Language Model based Autonomous Agents*. Frontiers of Computer Science (2024). <https://arxiv.org/abs/2308.11432>